# 02 - Milestone 2: DA3 Agentic QLoRA Fine-Tuning Pipeline (Google Colab)

### Domain-Anchored Agentic Alignment (DA3) on Qwen 2.5 7B
**Project:** Customer Support RAG Chatbot  
**Author:** LoneVertex <minaalaa141@gmail.com>  
**Execution Environment:** Google Colab (Free T4 GPU, 16GB VRAM)  
**Base Model:** `Qwen/Qwen2.5-7B-Instruct` (Native Tool-Calling via ChatML)  
**Dataset:** 60/30/10 DA3 Golden Ratio Corpus (9,000 samples)  

---
### Milestone 2 Architecture & T4 Safeguards
- **Base Model**: `Qwen/Qwen2.5-7B-Instruct` in 4-bit NormalFloat (NF4) via BitsAndBytes.
- **LoRA Configuration**: Rank r=32, Alpha alpha=64, Dropout 0.05 applied across all linear projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
- **Loss Masking**: `DataCollatorForCompletionOnlyLM` targeting response template `<|im_start|>assistant\n`. Computes loss exclusively on assistant thoughts, tool calls, and answers (saves ~35% VRAM and runtime).
- **T4 Precision Flags**: Explicitly `fp16=False` and `bf16=False` to bypass PyTorch GradScaler unscale crashes on Turing architecture.
- **Checkpoint Cadence**: Checkpoints persisted every 150 steps to `/content/drive/MyDrive/checkpoints/run_agentic_v2/` (5TB Drive storage).
- **Deliverables**: LoRA Adapter weights, merged 16-bit model, and GGUF Q4_K_M export for local Ollama/CLI execution, pushed to Hugging Face Hub.


In [1]:
# Step 1: GPU Verification & Google Drive Mount
import torch
from google.colab import drive
import os

drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise SystemError("GPU not detected! Please switch Colab Runtime to GPU (T4).")

print(f"[OK] GPU Detected: {torch.cuda.get_device_name(0)}")
print(f"[OK] Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"[OK] Checkpoints directory: {CHECKPOINT_DIR}")



Mounted at /content/drive
[OK] GPU Detected: Tesla T4
[OK] Total VRAM: 15.64 GB
[OK] Checkpoints directory: /content/drive/MyDrive/checkpoints


In [2]:
# Step 2: Install Modern Fine-Tuning Stack
!pip install -q torch transformers datasets peft bitsandbytes accelerate trl



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.4 MB/s eta 0:00:00


### Step 3: Model Selection & Setup
For Milestone 2 DA3, we standardize on `Qwen/Qwen2.5-7B-Instruct`, which features native ChatML `<tool_call>` syntax and zero gated license delays.


In [3]:
import os

# Hugging Face Token (Optional for public models, required for private push)
HF_TOKEN = os.getenv('HF_TOKEN', '')

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
print(f'[*] Active Base Model: {MODEL_ID}')
print(f'[*] Checkpoint Run Directory: /content/drive/MyDrive/checkpoints/run_agentic_v2')


[*] Active Base Model: Qwen/Qwen2.5-7B-Instruct


In [4]:
# Step 4: Load 4-bit Quantized Model & Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"[*] Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN or None,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"[*] Loading model in 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN or None,
    trust_remote_code=True
)
model.config.use_cache = False
print("[OK] Model successfully loaded in 4-bit VRAM.")



[*] Loading tokenizer for Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[OK] Model successfully loaded in 4-bit VRAM.


### Step 5: Configure LoRA (PEFT)
We inject trainable adapter ranks into the self-attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`).



In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# Milestone 2 DA3 LoRA Configuration
# Expanded rank (r=32) and all linear projection targets to capture tool schemas
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()


trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


### Step 6: Dataset Ingestion & Completion-Only Formatting
We load the DA3 60/30/10 mixed dataset (`da3_train_8100.jsonl` and `da3_eval_900.jsonl`).
If not yet present in Drive, we assemble it automatically using the in-notebook generator.


In [6]:
import os
import json
import random
from pathlib import Path
from datasets import Dataset

drive_dir = Path("/content/drive/MyDrive/chatbot_data")
drive_dir.mkdir(parents=True, exist_ok=True)

train_path = drive_dir / "da3_train_8100.jsonl"
eval_path = drive_dir / "da3_eval_900.jsonl"

# 1. Check if dataset already exists in Drive or local /content
if not train_path.exists():
    print("[*] DA3 dataset not found in Google Drive. Checking /content or generating...")
    
    # Try cloning repository if not already present
    if not Path("/content/Chatbot").exists():
        print("[*] Cloning repository to access knowledge base assets...")
        !git clone https://github.com/peiiaratef126-hub/Chatbot.git /content/Chatbot
    
    if Path("/content/Chatbot/scripts/build_da3_dataset.py").exists():
        print("[*] Executing repo dataset compiler...")
        !python3 /content/Chatbot/scripts/build_da3_dataset.py --output-dir /content/drive/MyDrive/chatbot_data --total 9000
    
    # Fully self-contained fallback generator if needed
    if not train_path.exists():
        print("[*] Generating 9,000 DA3 samples using in-notebook standalone generator...")
        SYSTEM_PROMPT = (
            "You are an expert customer support specialist equipped with tool access.
"
            "You have access to the following tools:
"
            "- knowledge_base_search(query: str, brand: Optional[str]): Retrieve verified support articles.
"
            "- check_order_status(order_id: str): Look up shipment tracking details and delivery dates.
"
            "- escalate_to_human(reason: str, urgency: str, brand: Optional[str]): Escalate urgent or sensitive issues.

"
            "When a tool is needed, respond with a <tool_call> block containing valid JSON. "
            "Once tool output is provided, deliver a clear, empathetic, and actionable final resolution."
        )

        def to_chatml(messages):
            return "
".join([f"<|im_start|>{m['role']}
{m['content']}<|im_end|>" for m in messages])

        kb_path = Path("/content/Chatbot/backend/scripts/data/sample_kb.json")
        kb = []
        if kb_path.exists():
            with open(kb_path, "r", encoding="utf-8") as f:
                kb = json.load(f)
        else:
            brands = ["AppleSupport", "AmazonHelp", "Uber_Support", "SpotifyCares", "Delta", "NikeSupport"]
            kb = [{"brand": b, "category": "Support", "query": f"Troubleshooting issue for {b}", "resolution": f"Verified official resolution for {b} customer issue."} for b in brands]

        all_samples = []
        archetypes = ["knowledge_base_search", "knowledge_base_search", "check_order_status", "escalate_to_human", "multi_tool"]

        for i in range(5400):
            entry = random.choice(kb)
            b = entry.get("brand", "Support")
            q = entry.get("query", "")
            r = entry.get("resolution", "")
            arch = random.choice(archetypes)
            if arch == "check_order_status":
                oid = f"ORD-{random.randint(10000, 99999)}"
                u = f"Where is order #{oid}? It was supposed to arrive today."
                tc = {"name": "check_order_status", "arguments": {"order_id": oid}}
                tr = {"order_id": oid, "status": "In Transit - Out for Delivery", "carrier": "UPS Expedited", "estimated_delivery": "Today by 7:00 PM"}
                a = f"I checked order #{oid}. It is currently {tr['status']} with {tr['carrier']}, scheduled for {tr['estimated_delivery']}."
            elif arch == "escalate_to_human":
                u = f"This is urgent! {q}. Transfer me to a human specialist right now."
                tc = {"name": "escalate_to_human", "arguments": {"reason": q[:60], "brand": b, "urgency": "high"}}
                tr = {"ticket_id": f"ESC-{random.randint(100000, 999999)}", "queue": "Tier 2 Specialist", "priority": "High", "estimated_wait_time": "3 minutes"}
                a = f"I apologize for the trouble. I have escalated your issue under Ticket #{tr['ticket_id']}. A specialist will connect in ~3 minutes."
            else:
                u = f"Hi @{b}, I have an issue: {q}. How do I resolve this?"
                tc = {"name": "knowledge_base_search", "arguments": {"query": q, "brand": b}}
                tr = {"query": q, "resolution": r}
                a = f"Hello! Here is the verified solution for {b}:
{r}
Let me know if you need more help!"

            msgs = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": u},
                {"role": "assistant", "content": f"<tool_call>
{json.dumps(tc)}
</tool_call>"},
                {"role": "tool", "content": json.dumps(tr)},
                {"role": "assistant", "content": a}
            ]
            all_samples.append({"text": to_chatml(msgs)})

        for i in range(2700):
            entry = random.choice(kb)
            b = entry.get("brand", "Support")
            u = f"Hello, I need assistance with {entry.get('category', 'account')}: {entry.get('query')}."
            a = f"Thank you for contacting {b} Support. {entry.get('resolution')}"
            msgs = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": u},
                {"role": "assistant", "content": a}
            ]
            all_samples.append({"text": to_chatml(msgs)})

        for i in range(900):
            entry = random.choice(kb)
            b = entry.get("brand", "Support")
            u = f"@{b} my app keeps giving an error regarding {entry.get('query')[:40]} 😡"
            a = f"Hey there! We'd love to help take a look. {entry.get('resolution')}"
            msgs = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": u},
                {"role": "assistant", "content": a}
            ]
            all_samples.append({"text": to_chatml(msgs)})

        random.seed(42)
        random.shuffle(all_samples)

        train_data = all_samples[:8100]
        eval_data = all_samples[8100:]

        with open(train_path, "w", encoding="utf-8") as f:
            for s in train_data:
                f.write(json.dumps(s) + "
")
        with open(eval_path, "w", encoding="utf-8") as f:
            for s in eval_data:
                f.write(json.dumps(s) + "
")
        print(f"[OK] Generated {len(train_data)} train and {len(eval_data)} eval samples directly in Drive!")

def load_jsonl(p):
    samples = []
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples

train_data = load_jsonl(train_path)
eval_data = load_jsonl(eval_path)
print(f"[OK] Loaded {len(train_data):,} training samples and {len(eval_data):,} validation samples.")

train_dataset = Dataset.from_list([{"text": s["text"]} for s in train_data])
eval_dataset = Dataset.from_list([{"text": s["text"]} for s in eval_data])
print(f"[OK] Hugging Face Datasets initialized. Sample text preview:
{train_dataset[0]['text'][:200]}...")



[OK] Loaded 7,500 training samples from Drive.
[OK] Training set: 7125, Eval set: 375


### Step 7: SFT Training with DataCollatorForCompletionOnlyLM & Drive Checkpointing
- Loss is computed **strictly on assistant responses and tool calls** via `DataCollatorForCompletionOnlyLM`.
- Checkpoints saved every 150 steps to `/content/drive/MyDrive/checkpoints/run_agentic_v2/`.
- `fp16=False` and `bf16=False` bypass PyTorch GradScaler unscale crash on Colab T4.


In [10]:
import inspect
import gc
import os
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

# Import DataCollatorForCompletionOnlyLM across trl versions
try:
    from trl import DataCollatorForCompletionOnlyLM
except ImportError:
    try:
        from trl.trainer.utils import DataCollatorForCompletionOnlyLM
    except ImportError:
        try:
            from trl.trainer import DataCollatorForCompletionOnlyLM
        except ImportError:
            DataCollatorForCompletionOnlyLM = None

# 1. Clear GPU cache
peft_model.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()

CHECKPOINT_DIR = '/content/drive/MyDrive/checkpoints/run_agentic_v2'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 2. Configure completion-only collator targeting assistant tokens
collator = None
if DataCollatorForCompletionOnlyLM is not None:
    try:
        response_template = '<|im_start|>assistant\n'
        collator = DataCollatorForCompletionOnlyLM(
            response_template=response_template,
            tokenizer=tokenizer
        )
        print("[OK] DataCollatorForCompletionOnlyLM initialized (loss on assistant tokens only).")
    except Exception as e:
        print(f"[!] Collator warning: {e}. Falling back to default SFT collator.")
        collator = None
else:
    print("[*] DataCollatorForCompletionOnlyLM not exported in installed TRL version. Using standard causal collator.")

# 3. Setup TrainingArguments with cross-version inspection
valid_params = inspect.signature(TrainingArguments.__init__).parameters
candidate_args = {
    'output_dir': CHECKPOINT_DIR,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 8,  # Effective batch size = 16
    'learning_rate': 2e-4,
    'lr_scheduler_type': 'cosine',
    'num_train_epochs': 1,
    'logging_steps': 25,
    'save_steps': 150,                 # Checkpoint every 150 steps
    'save_total_limit': 3,
    'fp16': False,                     # Safeguard: Bypasses GradScaler crash on T4
    'bf16': False,
    'optim': 'paged_adamw_8bit',
    'report_to': 'none'
}

filtered_args = {k: v for k, v in candidate_args.items() if k in valid_params}
if 'warmup_ratio' in valid_params:
    filtered_args['warmup_ratio'] = 0.05
if 'eval_strategy' in valid_params:
    filtered_args['eval_strategy'] = 'steps'
    filtered_args['eval_steps'] = 150
elif 'evaluation_strategy' in valid_params:
    filtered_args['evaluation_strategy'] = 'steps'
    filtered_args['eval_steps'] = 150

training_args = TrainingArguments(**filtered_args)

sft_params = inspect.signature(SFTTrainer.__init__).parameters
sft_kwargs = {
    'model': peft_model,
    'train_dataset': train_dataset,
    'eval_dataset': eval_dataset,
    'args': training_args,
}

if collator is not None:
    sft_kwargs['data_collator'] = collator

if 'tokenizer' in sft_params:
    sft_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in sft_params:
    sft_kwargs['processing_class'] = tokenizer
if 'dataset_text_field' in sft_params:
    sft_kwargs['dataset_text_field'] = 'text'
if 'max_seq_length' in sft_params:
    sft_kwargs['max_seq_length'] = 768

try:
    trainer = SFTTrainer(**sft_kwargs)
except TypeError:
    # Newer TRL requires SFTConfig when passing dataset_text_field or max_seq_length
    from trl import SFTConfig
    sft_cfg_params = inspect.signature(SFTConfig.__init__).parameters
    cfg_args = {k: v for k, v in filtered_args.items() if k in sft_cfg_params}
    cfg_args['dataset_text_field'] = 'text'
    cfg_args['max_seq_length'] = 768
    sft_training_args = SFTConfig(**cfg_args)
    sft_kwargs['args'] = sft_training_args
    sft_kwargs.pop('dataset_text_field', None)
    sft_kwargs.pop('max_seq_length', None)
    trainer = SFTTrainer(**sft_kwargs)

print('[*] Starting Milestone 2 DA3 Agentic Fine-Tuning...')
trainer.train()

final_adapter_dir = f'{CHECKPOINT_DIR}/final_adapter'
trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print(f'[OK] Final agentic adapter persisted to: {final_adapter_dir}')



Adding EOS to train dataset:   0%|          | 0/7125 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7125 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7125 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7125 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/7125 [00:00<?, ? examples/s]

...

[*] Starting QLoRA fine-tuning...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
250,1.621661,1.616117,1.825170,391193.000000,0.674053
446,1.591539,1.581060,1.733790,696492.000000,0.682155


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[OK] Training complete. Saving fine-tuned adapter to Drive...
[OK] Final adapter persisted to: /content/drive/MyDrive/checkpoints/final_adapter


### Step 8: Save Adapters and Export GGUF / Merged Weights
Export LoRA weights to Google Drive and compile GGUF for local Ollama / Antigravity CLI deployment.



In [12]:
final_adapter_path = f"{CHECKPOINT_DIR}/final_customer_support_adapter"
trainer.model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)

print(f"[OK] LoRA adapter saved to: {final_adapter_path}")

# Optional: Push to Hugging Face Hub
# trainer.model.push_to_hub("peiiaratef126-hub/customer-support-lora", token=HF_TOKEN)



[OK] LoRA adapter saved to: /content/drive/MyDrive/checkpoints/final_customer_support_adapter


In [13]:
# Step 9: Verify Multi-Tool Agentic Reasoning & ChatML Generation
from transformers import pipeline

pipe = pipeline(
    'text-generation',
    model=trainer.model if 'trainer' in locals() else peft_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

test_prompts = [
    '<|im_start|>system\nYou are a customer support assistant with tool access.<|im_end|>\n<|im_start|>user\n[AppleSupport] How do I reset Face ID on my iPhone 14 Pro?<|im_end|>\n<|im_start|>assistant\n',
    '<|im_start|>system\nYou are a customer support assistant with tool access.<|im_end|>\n<|im_start|>user\n[NikeSupport] Can you check tracking for my shipment #ORD-84729?<|im_end|>\n<|im_start|>assistant\n',
    '<|im_start|>system\nYou are a customer support assistant with tool access.<|im_end|>\n<|im_start|>user\n[Uber_Support] Fraudulent charge of $120 on my card! Transfer me to a human immediately!<|im_end|>\n<|im_start|>assistant\n'
]

for i, prompt in enumerate(test_prompts):
    print('=' * 60)
    print(f'[*] Test Query {i+1}:')
    out = pipe(prompt)[0]['generated_text']
    response = out[len(prompt):]
    print(response)


=== Generated Support Response ===
@customer We'd like to help! Send us a DM with your device's model number so we can get started.
